<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/10_grasp_learning_ppo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# HSR Grasp Learning with PPO + IK Curriculum / PPO + IKカリキュラムによる把持学習

**目的 / Objective:**

- **PPO** (Proximal Policy Optimization) を用いて、IK把持パイプラインの上に**残差ポリシー**を学習する方法を学ぶ / Learn to train a **residual policy** on top of the IK pick pipeline using **PPO**.
- 3段階のカリキュラムで、IK主導からポリシー主導へと徐々に制御を移行する / Gradually shift control from IK-dominant to policy-dominant via a 3-stage curriculum.
- stable-baselines3 PPOとGenesis並列シミュレーションを組み合わせた強化学習を体験する / Experience reinforcement learning combining stable-baselines3 PPO with Genesis parallel simulation.

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better). Training takes ~5 min for 20K steps on a T4 with 16 envs.

## 概要 / Overview

前回のチュートリアル (Section 8) では、CMA-ESで把持パラメータを最適化しました。このノートブックでは一歩進んで、**ニューラルネットポリシー**がIK軌跡の上に残差補正を学習します。

In the previous tutorial (Section 8), we optimized grasp parameters with CMA-ES. This notebook goes one step further: a **neural network policy** learns residual corrections on top of IK trajectories.

### アーキテクチャ / Architecture

![PPO residual policy architecture](https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/examples/tutorials/img/ppo_architecture.svg)

ポリシーは9次元の残差行動を出力し、IKが計算した目標位置に加算されます。カリキュラムの `policy_weight` が0から0.7へ増加するにつれ、ポリシーの補正量が大きくなります。

The policy outputs a 9D residual action that is added to IK-computed targets. As the curriculum `policy_weight` increases from 0 to 0.7, the policy's corrections grow larger.

### 行動空間 / Action space (9D)

| Component | Dims | Scale | Description |
|-----------|------|-------|-------------|
| `delta_arm` | 5 | 0.1 rad | Arm joint residual / 腕関節残差 |
| `delta_base` | 3 | 0.05 m / 0.1 rad | Base xy-yaw residual / 台車xy-yaw残差 |
| `gripper_effort` | 1 | 4.0 N + 2.0 | Gripper force [0, 8] N / グリッパ力 |

### カリキュラム / Curriculum (3 stages)

| Stage | IK weight | Policy weight | Description |
|-------|-----------|---------------|-------------|
| 0 | 1.0 | 0.0 | Pure IK warmup (policy observes) / IKのみ（ポリシーは観測のみ） |
| 1 | 0.7 | 0.3 | 70% IK + 30% policy blend / 70% IK + 30% ポリシー |
| 2 | 0.3 | 0.7 | Policy-dominant (terminal) / ポリシー主導（最終段階） |

## 1. Setup / セットアップ

依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
Install dependencies, clone the repo, and configure GPU rendering.

In [ ]:
import importlib, urllib.request

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering.
setup_colab()

# Install stable-baselines3 for PPO
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'stable-baselines3', '-q'], check=True)
print('stable-baselines3 installed.')

In [ ]:
import sys, pathlib

# Add repo to path (same as other tutorials)
REPO_DIR = pathlib.Path('/content/hsr-genesis')
SRC_DIR = str(REPO_DIR / 'src')
EXAMPLES_DIR = str(REPO_DIR / 'examples' / 'rl')
for p in [SRC_DIR, EXAMPLES_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import genesis as gs
if not getattr(gs, '_initialized', False):
    gs.init(backend=gs.gpu)
else:
    print('Genesis already initialized.')

import torch
import numpy as np
print(f'Genesis device: {gs.device}')

## 2. Curriculum Manager / カリキュラム管理

3段階のカリキュラムを管理する `CurriculumManager` を確認します。各段階は成功率の閾値に基づいて自動的に進行します。

Inspect the `CurriculumManager` that handles the 3-stage curriculum. Each stage advances automatically based on success rate thresholds.

In [ ]:
from curriculum import CurriculumManager, STAGES, EVAL_INTERVAL

print('Curriculum stages:')
print(f'{"Stage":>5} {"IK_w":>6} {"Policy_w":>8} {"Threshold":>10} {"Warmup":>8}')
print('-' * 45)
for i, s in enumerate(STAGES):
    print(f'{i:>5} {s.ik_weight:>6.1f} {s.policy_weight:>8.1f} '
          f'{s.advance_threshold:>10.2f} {s.warmup_steps:>8d}')
print(f'\nEval interval: every {EVAL_INTERVAL:,} sim steps')

# Demo: simulate curriculum advancement
mgr = CurriculumManager()
print(f'\nInitial: stage={mgr.stage}, policy_weight={mgr.policy_weight}')
mgr.update(0.55, 10000)  # After warmup, threshold=0.0 so always advances
print(f'After 1st eval: stage={mgr.stage}, policy_weight={mgr.policy_weight}')
mgr.update(0.60, 20000)  # Stage 1 needs 50% for 2 consecutive
print(f'After 2nd eval: stage={mgr.stage}, policy_weight={mgr.policy_weight}')
mgr.update(0.55, 30000)
print(f'After 3rd eval: stage={mgr.stage}, policy_weight={mgr.policy_weight}')

## 3. IK Planner / IKプランナー

`IKPlanner` は各エピソードの開始時に3つのIK解（アプローチ、降下、持ち上げ）を計算します。CMA-ESで最適化された把持パラメータを使用します。

The `IKPlanner` computes 3 IK solutions (approach, descend, lift) at the start of each episode. It uses CMA-ES optimized grasp parameters.

In [ ]:
from ik_planner import (
    IKPlanner, IKPlan,
    APPROACH_STEPS, DESCEND_STEPS, GRASP_STEPS, LIFT_STEPS,
    APPROACH_START, DESCEND_START, GRASP_START, LIFT_START,
    MAX_STEPS, PRE_GRASP_HEIGHT, GRASP_OFFSET_Z, LIFT_HEIGHT,
    get_object_params,
)

print('Phase structure (900 steps = 18s at dt=0.02):')
print(f'  Approach: steps {APPROACH_START}-{DESCEND_START-1} ({APPROACH_STEPS} steps, {APPROACH_STEPS*0.02:.1f}s)')
print(f'  Descend:  steps {DESCEND_START}-{GRASP_START-1} ({DESCEND_STEPS} steps, {DESCEND_STEPS*0.02:.1f}s)')
print(f'  Grasp:    steps {GRASP_START}-{LIFT_START-1} ({GRASP_STEPS} steps, {GRASP_STEPS*0.02:.1f}s)')
print(f'  Lift:     steps {LIFT_START}-{MAX_STEPS-1} ({LIFT_STEPS} steps, {LIFT_STEPS*0.02:.1f}s)')
print(f'  Total:    {MAX_STEPS} steps ({MAX_STEPS*0.02:.1f}s)')
print()

# Show CMA-ES optimized params for apple (our PPO training object)
params = get_object_params('ycb_013_apple')
print(f'CMA-ES params for apple:')
for k, v in params.items():
    print(f'  {k:25s} {v:.4f}')

## 4. RL Environment / RL環境

`HSRPickRLEnv` はN個のGenesis並列環境を1つのGymnasium環境としてラップします。ポリシーは32次元の観測から9次元の残差行動を出力します。

`HSRPickRLEnv` wraps N Genesis parallel envs as a single Gymnasium env. The policy outputs a 9D residual action from a 32D observation.

In [ ]:
from hsr_pick_rl_env import HSRPickRLEnv, BatchedGenesisVecEnv, OBS_DIM, ACTION_DIM

print(f'Observation dim: {OBS_DIM}')
print(f'Action dim:      {ACTION_DIM}')
print()
print('Observation components (32D):')
print('  Object pos (robot frame)  [3]')
print('  Object yaw (robot frame)  [1]')
print('  End-effector pos          [3]')
print('  Arm joint positions       [5]')
print('  Base xy-yaw               [3]')
print('  Gripper motor position    [1]')
print('  Curriculum policy_weight  [1]')
print('  Step progress             [1]')
print('  Previous action           [9]')
print('  Phase one-hot (5 phases)  [5]')
print('  Total:                     32')
print()
print('Action components (9D, tanh-squashed):')
print('  delta_arm   [5]  x 0.1 rad  -> added to IK arm target')
print('  delta_base  [3]  x 0.05 m   -> added to IK base target')
print('  gripper_eff [1]  x 4.0+2.0 N -> gripper force [0, 8] N')

## 5. Baseline Evaluation (Pure IK) / ベースライン評価（IKのみ）

カリキュラムStage 0（ポリシーweight=0、純IK）での成功率を測定します。これがPPO学習の開始点となります。

Measure success rate at curriculum Stage 0 (policy_weight=0, pure IK). This is the PPO training starting point.

In [ ]:
N_ENVS = 16
SETTLE_STEPS = 30
OBJECT = 'ycb_013_apple'  # apple: 33% baseline — more room for PPO to improve than foam_brick (80%)

# Create env at stage 0 (pure IK)
curriculum = CurriculumManager()
env = HSRPickRLEnv(
    n_envs=N_ENVS,
    object_name=OBJECT,
    seed=42,
    settle_steps=SETTLE_STEPS,
    curriculum=curriculum,
)

# Run one episode with zero action (pure IK)
obs, info = env.reset()
print(f'IK success per phase: {info["ik_success"].sum(axis=0)}')
print(f'CMA-ES params: pre_grasp={env._plan.pre_grasp_height:.3f}, '
      f'offset_z={env._plan.grasp_offset_z:.3f}, effort={env._plan.gripper_effort:.2f}')
print()

import numpy as np
for i in range(MAX_STEPS):
    action = np.zeros((N_ENVS, ACTION_DIM), dtype=np.float32)
    obs, reward, term, trunc, info = env.step(action)
    if i % 100 == 0:
        print(f'  step {i:>3d}: success={info["success"].sum()}/{N_ENVS}')
    if term.all() or trunc.all():
        break

baseline_rate = info['success'].sum() / N_ENVS
print(f'\nBaseline (pure IK) success rate: {baseline_rate:.1%}')

## 6. PPO Training / PPO学習

stable-baselines3のPPOで残差ポリシーを学習します。カリキュラムコールバックが10,000ステップごとに評価を行い、成功率に基づいて段階を進行させます。

Train the residual policy with stable-baselines3 PPO. A curriculum callback runs evaluation every 10,000 steps and advances stages based on success rate.

### PPOの仕組み / How PPO works

1. **ロールアウト / Rollout**: 現在のポリシーで2048ステップ分のデータを収集 / Collect 2048 steps of data with current policy
2. **GAE**: Generalized Advantage Estimation でアドバンテージを計算 / Compute advantages with GAE
3. **クリッピング / Clipping**: ポリシー更新を ±20% に制限して安定化 / Clamp policy ratio to ±20% for stability
4. **エントロピー / Entropy**: 探索を促進するエントロピーボーナス / Entropy bonus to encourage exploration

![PPO algorithm overview](https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/examples/tutorials/img/ppo_algorithm.svg)

> **Tip:** Colab T4では16環境・40Kステップで約10分かかります。ローカルGPUで256環境・500Kステップを実行するとより良い結果が得られます / On Colab T4, 16 envs for 40K steps takes ~10 min. Run 256 envs for 500K steps on a local GPU for better results.

### Q-learning / DQN との比較 / From the Q-learning / DQN perspective

PPO と DQN はどちらも**モデルフリー**な強化学習手法ですが、学習する対象と行動の選び方が異なります。ここではすでに馴染みがあるかもしれない Q-learning / DQN と対比しながら PPO を理解します。

PPO and DQN are both **model-free** RL methods, but they differ in *what they learn* and *how they act*. If you already know Q-learning / DQN, this section builds the bridge.

#### 学習する対象 / What each method learns

| | Q-learning / DQN | PPO |
|---|---|---|
| 学ぶもの / Learns | 行動価値関数 Q(s,a) / Action-value function | ポリシー π(a\|s) + 価値 V(s) / Policy + value |
| 種類 / Type | 価値ベース / Value-based | ポリシベース（Actor-Critic）/ Policy-based (actor-critic) |
| 行動空間 / Action space | 離散のみ / Discrete only | 離散でも連続でも / Discrete or continuous |
| 出力 / Output | (s,a) → 期待リターンの表/ネットワーク | 行動分布を出力するネットワーク |
| 行動の選び方 / How to act | `a = argmax_a Q(s,a)` | `a ~ π(·|s)` からサンプリング / Sample |

DQN は「各状態で各行動がどれくらい良いか」を学び、最も良い行動を選びます。PPO は「どのような行動分布からサンプリングすべきか」を学び、そこからサンプリングします。

DQN learns "how good is each action in this state" and picks the best. PPO learns "what distribution should I sample actions from" and samples.

#### Q-learning の世界観 / The Q-learning worldview

Q-learning の更新ターゲット（Bellman バックアップ）は次のとおりです:

Q-learning's update target (the Bellman backup):

```
Q(s,a) <- Q(s,a) + alpha * [ r + gamma * max_a' Q(s',a') - Q(s,a) ]
                                       ^^^^^^^^^^^^^^^^^^^
                                       bootstrapped target
```

DQN はテーブルをニューラルネットワーク Q_θ(s,a) で置き換え、次の工夫を使います:

DQN replaces the table with a neural network Q_θ(s,a) and uses:

- **経験再生 / Experience replay** — 連続サンプルの相関を崩す / Break correlation in sequential samples
- **ターゲットネットワーク / Target network** Q_θ⁻ — ブートストラップターゲットを安定化 / Stabilize the bootstrapped target
- **損失 / Loss**: `MSE(Q_θ(s,a), r + gamma * max_a' Q_θ⁻(s',a'))`

重要なポイント: **ターゲット自体が次の状態の Q 値に対する現在のポリシーの貪欲な選択に依存しています**。これが DQN にターゲットネットワークが必要な理由です — これがないと `max` 演算子が動く標的を追い続けて発散します。

Key idea: **the target itself depends on the current policy's greedy choice over the next state's Q-values**. This is why DQN needs a target network — without it, the `max` operator chases a moving target and diverges.

#### ポリシーグラディエントの世界観 / The policy-gradient worldview (where PPO lives)

Q を学んでからポリシーを導くのではなく、ポリシーを直接学びます。ポリシーグラディエント定理は次のとおりです:

Instead of learning Q and deriving a policy, learn the policy directly. The policy gradient theorem:

```
grad J(theta) = E[ grad log pi_theta(a|s) * G_t ]
                                       ^^^
                                       return (or advantage)
```

直感: 状態 `s` で行動 `a` が高いリターンをもたらしたら log π_θ(a\|s) を増やす — その行動をより起こりやすくする。低いリターンなら減らす。

Intuition: if action `a` in state `s` led to a high return, increase log π_θ(a\|s) — make that action more likely. If low return, decrease it.

**素のポリシーグラディエントの問題 / Problems with vanilla policy gradient:**
- 分散が高い（G_t がノイズ多い）/ High variance (G_t is noisy)
- 1回の悪い更新でポリシーが壊れる（移動量に制限がない）/ A single bad update can destroy the policy

#### PPO = ポリシーグラディエント + 2つの修正 / PPO = policy gradient + two fixes

**修正1: 生リターンではなくアドバンテージを使う / Use the advantage, not the raw return** (Actor-Critic 部分 — 価値ベース手法への橋渡し)

```
A_t = G_t - V_phi(s_t)
```

`V_φ(s)` は学習した価値関数（「クリティック」）— DQN が学ぶのと同じ種類のオブジェクトですが、行動価値ではなく状態価値を推定します。これをベースラインとして引くことで分散が減ります。これが **GAE** (Generalized Advantage Estimation) です:

`V_φ(s)` is a learned value function (the "critic") — the same kind of object DQN learns, except it estimates state value, not action value. Subtracting it as a baseline reduces variance. This is **GAE**:

```
delta_t = r_t + gamma * V(s_{t+1}) - V(s_t)        <- TD error (Q-learning update に似ている!)
A_t     = delta_t + (gamma*lambda)*delta_{t+1} + (gamma*lambda)^2 * delta_{t+2} + ...
```

`δ_t` は文字通り Bellman 残差 — Q-learning が最小化するのと同じ量です。PPO はそれを Q テーブルの更新に使うのではなく、ポリシーグラディエントのための分散減少シグナルとして使います。

`δ_t` is literally the Bellman residual — the same quantity Q-learning minimizes. PPO just uses it as a variance-reduced signal for the policy gradient, instead of using it to update a Q-table.

**修正2: ポリシー更新を制限する / Clamp the policy update** (PPO の定義的特徴 — 「Proximal」の部分)

新旧ポリシーの比がどれだけ動いたかを示します:

The ratio between new and old policy tells you how much you've moved:

```
ratio = pi_theta(a|s) / pi_theta_old(a|s)
```

- ratio = 1 → 変化なし / no change
- ratio > 1 → 行動が起こりやすくなった / action became more likely
- ratio < 1 → 行動が起こりにくくなった / action became less likely

PPO のクリップ目的関数 / PPO's clipped objective:

```
L_clip = E[ min( ratio * A,  clip(ratio, 1+-eps) * A ) ]
```

アドバンテージが正（行動が良かった）なら確率を増やしたいが、`clip(ratio, 1+eps)` が増加を 1+eps（例えば 1.2）で止めます。アドバンテージが負なら `clip(ratio, 1-eps)` が減少を止めます。**ポリシーは1回の更新で古いポリシーから eps 以上離れられません**、アドバンテージ信号がどれだけ大きくても。

If the advantage is positive (action was good), you want to increase its probability — but `clip(ratio, 1+eps)` caps the increase at 1+eps (e.g. 1.2). If the advantage is negative, `clip(ratio, 1-eps)` caps the decrease. **The policy cannot move more than eps away from the old policy per update**, no matter how large the advantage signal.

これは DQN のターゲットネットワークの概念的な類似物です: どちらも最適化が動く標的を追って発散するのを防ぐために存在します。ただし仕組みは異なります:

This is the conceptual analogue of DQN's target network: both exist to prevent the optimization from chasing a moving target and diverging. But the mechanism is different:

- DQN は古いネットワークのコピーでターゲットを固定 / DQN freezes the target with an old network copy
- PPO はクリッピングでステップサイズを制約 / PPO constrains the step size via clipping

#### 学習ループの比較 / Training loop side-by-side

```
DQN                                 PPO
----                                ---
1. (s,a,r,s') を eps-greedy で収集  1. T ステップを pi_theta で収集 (on-policy)
2. リプレイバッファに保存           2. (バッファなし — on-policy, 使い捨て)
3. ミニバッチをサンプリング         3. ロールアウト全体の GAE アドバンテージ A_t を計算
4. target = r + gamma*max_a' Q⁻     4. ratio = pi_theta(a|s) / pi_theta_old(a|s)
5. Loss = MSE(Q_theta, target)      5. Loss = -clip(ratio,1+-eps)*A + c1*L_VF - c2*H
6. Q_theta を更新                   6. 同じロールアウトで K エポック更新
7. 定期的に theta -> theta-         7. (ターゲットネット不要 — クリップで安定化)
```

#### なぜこのタスクに PPO なのか / Why PPO for this task

このノートブックの行動空間は **9次元の連続値** です（`delta_arm` 5D + `delta_base` 3D + `gripper_effort` 1D）。DQN は本質的に連続行動を扱えません — DDPG/SAC が必要になるか、各次元を離散化する必要があります（次元の呪い: 各次元5ビンでも 5^9 ≈ 200万行動）。PPO は 9D 行動上のガウス分布を出力してそこからサンプリングするので、連続制御に自然です。これが本ノートブックで DQN ではなく PPO を使う理由です。

The action space here is **9D continuous** (`delta_arm` 5D + `delta_base` 3D + `gripper_effort` 1D). DQN fundamentally cannot handle continuous actions — you'd need DDPG/SAC, or discretize each dimension (curse of dimensionality: even 5 bins per dim = 5^9 ≈ 2M actions). PPO outputs a Gaussian distribution over the 9D action and samples from it, which is natural for continuous control. That's why this notebook uses PPO rather than DQN.

#### 1行まとめ / One-line summary

> **DQN** は「どの行動が最善か」を Q 値として学び貪欲に選ぶ。安定性は固定されたターゲットネットワークから来る。**PPO** は「どの行動分布が最善か」をポリシーとして学びサンプリングする。安定性はポリシー更新のクリッピングから来る。どちらも Bellman 残差を使う — DQN は回帰ターゲットとして、PPO はポリシーグラディエントのアドバンテージシグナルとして。

> **DQN** learns "which action is best" (Q-values) and acts greedily; stability comes from a frozen target network. **PPO** learns "which action distribution is best" (policy) and acts by sampling; stability comes from clipping the policy update. Both use the Bellman residual — DQN as the regression target, PPO as the advantage signal for the policy gradient.

In [ ]:
import time
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

OUTPUT_DIR = pathlib.Path('/content/ppo_ik_results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Fresh curriculum for training
train_curriculum = CurriculumManager()

# Create training env
train_env = HSRPickRLEnv(
    n_envs=N_ENVS,
    object_name=OBJECT,
    seed=0,
    settle_steps=SETTLE_STEPS,
    curriculum=train_curriculum,
)
vec_env = BatchedGenesisVecEnv(train_env)

# Curriculum callback
class CurriculumCallback(BaseCallback):
    def __init__(self, curriculum, eval_episodes=2, verbose=1):
        super().__init__(verbose)
        self.curriculum = curriculum
        self.eval_episodes = eval_episodes

    def _on_step(self):
        if self.curriculum.should_eval(self.num_timesteps):
            self._run_eval()
        return True

    def _run_eval(self):
        env = self.model.get_env()
        success_rates = []
        for _ in range(self.eval_episodes):
            obs = env.reset()
            done = False
            steps = 0
            while not done and steps < MAX_STEPS:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, dones, infos = env.step(action)
                done = bool(np.all(dones))
                steps += 1
            for info in infos:
                success_rates.append(float(info.get('success', False)))
        mean_success = float(np.mean(success_rates)) if success_rates else 0.0
        self.curriculum.update(mean_success, self.num_timesteps)
        if self.verbose:
            print(f'[eval] step={self.num_timesteps} success={mean_success:.3f} '
                  f'stage={self.curriculum.stage} pw={self.curriculum.policy_weight:.1f}')

callback = CurriculumCallback(train_curriculum, eval_episodes=2)

# PPO model — reduced n_steps and total_steps for faster Colab execution
model = PPO(
    'MlpPolicy',
    vec_env,
    learning_rate=3e-4,
    n_steps=1024,       # was 2048 — shorter rollouts, more frequent updates
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=None,
    policy_kwargs={'net_arch': [256, 256]},
    seed=0,
    device='auto',
)

TOTAL_STEPS = 20000  # was 40000 — ~5 min on Colab T4; increase to 500K on local GPU
print(f'\n[train] Starting PPO: {TOTAL_STEPS} steps, {N_ENVS} envs, object={OBJECT}')
print(f'[train] Curriculum: 3 stages, policy_weight = [0.0, 0.3, 0.7]')
print(f'[train] Episode length: {MAX_STEPS} steps ({MAX_STEPS*0.02:.1f}s)')
print()

t0 = time.time()
model.learn(total_timesteps=TOTAL_STEPS, callback=callback)
dt = time.time() - t0
print(f'\n[train] Training complete in {dt:.1f}s')
print(f'[train] Final stage: {train_curriculum.stage}, '
      f'policy_weight: {train_curriculum.policy_weight:.1f}')

# Save model + curriculum
model.save(str(OUTPUT_DIR / 'ppo_ik_curriculum'))
train_curriculum.save(str(OUTPUT_DIR / 'curriculum_state.json'))
print(f'[train] Model saved to {OUTPUT_DIR / "ppo_ik_curriculum.zip"}')
print(f'[train] Curriculum saved to {OUTPUT_DIR / "curriculum_state.json"}')

## 7. Training Analysis / 学習分析

カリキュラムの進行履歴と成功率の推移を可視化します。
Visualize curriculum progression and success rate over training.

In [ ]:
import matplotlib.pyplot as plt

history = train_curriculum.eval_history
if history:
    steps = [h['step'] for h in history]
    successes = [h['success_rate'] for h in history]
    stages = [h['stage'] for h in history]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    # Success rate
    ax1.plot(steps, successes, 'o-', color='tab:blue', label='Eval success')
    ax1.axhline(y=baseline_rate, color='tab:red', linestyle=':',
                label=f'IK baseline ({baseline_rate:.1%})')
    ax1.set_ylabel('Success rate')
    ax1.set_title('PPO + IK Curriculum Training Progress')
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)

    # Curriculum stage
    ax2.step(steps, stages, where='post', color='tab:green', linewidth=2)
    ax2.set_ylabel('Curriculum stage')
    ax2.set_xlabel('Training step')
    ax2.set_yticks([0, 1, 2])
    ax2.set_yticklabels(['Stage 0\n(pure IK)', 'Stage 1\n(70/30)', 'Stage 2\n(30/70)'])
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No eval history available.')

## 8. Policy Evaluation / ポリシー評価

学習済みポリシーをカリキュラム最終段階（policy_weight=0.7）で評価します。
Evaluate the trained policy at the final curriculum stage (policy_weight=0.7).

In [ ]:
# Load trained model
from stable_baselines3 import PPO as PPOLoad
from curriculum import CurriculumManager as CMLoad

model_loaded = PPOLoad.load(str(OUTPUT_DIR / 'ppo_ik_curriculum'), device='auto')
curriculum_loaded = CMLoad.load(str(OUTPUT_DIR / 'curriculum_state.json'))
print(f'Loaded model: stage={curriculum_loaded.stage}, '
      f'policy_weight={curriculum_loaded.policy_weight:.1f}')
print()

# Evaluate with trained curriculum
eval_env = HSRPickRLEnv(
    n_envs=N_ENVS,
    object_name=OBJECT,
    seed=12345,
    settle_steps=SETTLE_STEPS,
    curriculum=curriculum_loaded,
)
eval_vec = BatchedGenesisVecEnv(eval_env)

N_TRIALS = 5
trial_rates = []
print(f'{"Trial":>5} {"Steps":>6} {"Success":>8} {"Rate":>8}')
print('-' * 35)

for trial in range(N_TRIALS):
    obs = eval_vec.reset()
    done = False
    steps = 0
    while not done and steps < MAX_STEPS:
        action, _ = model_loaded.predict(obs, deterministic=True)
        obs, reward, dones, infos = eval_vec.step(action)
        done = bool(np.all(dones))
        steps += 1
    n_success = sum(1 for info in infos if info.get('success', False))
    rate = n_success / N_ENVS
    trial_rates.append(rate)
    print(f'{trial:>5} {steps:>6} {n_success:>5}/{N_ENVS:<2d} {rate:>8.1%}')

learned_rate = np.mean(trial_rates)
print(f'\n{"Mean":>5} {"":>6} {"":>8} {learned_rate:>8.1%}')
print(f'\nBaseline (pure IK):  {baseline_rate:.1%}')
print(f'Learned (PPO + IK):  {learned_rate:.1%}')
print(f'Improvement:         {learned_rate - baseline_rate:+.1%}')

## 9. Stage Comparison / 段階比較

カリキュラムの各段階（Stage 0, 1, 2）での成功率を比較し、ポリシーが徐々に制御を引き継ぐ様子を確認します。

Compare success rates at each curriculum stage (0, 1, 2) to see how the policy gradually takes over control.

In [ ]:
# Use cached eval results from training instead of running new episodes.
# train_curriculum.eval_history contains {step, stage, success_rate} per eval round.
stage_labels = ['Stage 0 (pure IK)', 'Stage 1 (70/30)', 'Stage 2 (30/70)']

# Extract best success rate per stage from training eval history
stage_rates = {}
for h in train_curriculum.eval_history:
    s = h['stage']
    if s not in stage_rates or h['success_rate'] > stage_rates[s]:
        stage_rates[s] = h['success_rate']

# Fallback: if a stage was never evaluated (e.g. training ended early),
# use baseline_rate for stage 0 and skip missing stages
if 0 not in stage_rates:
    stage_rates[0] = float(baseline_rate)

print('Stage success rates (from training eval history):')
for s in sorted(stage_rates):
    print(f'  {stage_labels[s]:>20s}: {stage_rates[s]:.1%}')

print()
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
stages_present = sorted(stage_rates)
ax.bar(range(len(stages_present)), [stage_rates[s] for s in stages_present],
       color=[['tab:red', 'tab:orange', 'tab:blue'][s] for s in stages_present], alpha=0.8)
ax.set_xticks(range(len(stages_present)))
ax.set_xticklabels([stage_labels[s].replace('Stage ', 'Stage\n') for s in stages_present])
ax.set_ylabel('Success rate')
ax.set_title(f'Policy Performance by Curriculum Stage ({OBJECT})')
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.3)
for i, s in enumerate(stages_present):
    v = stage_rates[s]
    ax.text(i, v + 0.02, f'{v:.0%}', ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()

## 10. Save & Load / 保存と読み込み

学習済みモデルとカリキュラム状態の保存・読み込み方法を示します。
Show how to save and load the trained model and curriculum state.

In [ ]:
import json

print('Saved files:')
print(f'  Model:      {OUTPUT_DIR / "ppo_ik_curriculum.zip"}')
print(f'  Curriculum: {OUTPUT_DIR / "curriculum_state.json"}')
print()

with open(OUTPUT_DIR / 'curriculum_state.json') as f:
    ckpt = json.load(f)
print(f'Curriculum state:')
print(f'  Final stage:       {ckpt["stage_idx"]}')
print(f'  Eval history:      {len(ckpt["eval_history"])} rounds')
if ckpt['eval_history']:
    last = ckpt['eval_history'][-1]
    print(f'  Last eval:         step={last["step"]}, success={last["success_rate"]:.3f}')
print()
print('To load in your own script:')
print('''
  from stable_baselines3 import PPO
  from curriculum import CurriculumManager
  from hsr_pick_rl_env import HSRPickRLEnv, BatchedGenesisVecEnv

  model = PPO.load("ppo_ik_curriculum.zip")
  curriculum = CurriculumManager.load("curriculum_state.json")

  env = HSRPickRLEnv(n_envs=16, object_name="ycb_013_apple",
                    curriculum=curriculum, seed=42, settle_steps=30)
  vec_env = BatchedGenesisVecEnv(env)
  obs = vec_env.reset()
  action, _ = model.predict(obs, deterministic=True)
  obs, reward, dones, infos = vec_env.step(action)
''')

## 11. Command-Line Usage / コマンドライン使用法

ノートブック外でスクリプトとして実行する方法を示します。
Show how to run the training and evaluation as command-line scripts.

In [ ]:
print('Command-line usage:')
print()
print('# Train (16 envs, 20K steps)')
print('PYTHONPATH=src .venv/bin/python examples/rl/train_ppo_ik_curriculum.py \\')
print('    --envs 16 --total-steps 20000 --object ycb_013_apple')
print()
print('# Quick test (8 envs, 2K steps)')
print('PYTHONPATH=src .venv/bin/python examples/rl/train_ppo_ik_curriculum.py \\')
print('    --envs 8 --total-steps 2000 --settle-steps 30')
print()
print('# Evaluate trained model')
print('PYTHONPATH=src .venv/bin/python examples/rl/eval_ppo_policy.py \\')
print('    --model results/ppo_ik_curriculum/ppo_ik_curriculum.zip \\')
print('    --curriculum results/ppo_ik_curriculum/curriculum_state.json \\')
print('    --envs 32 --trials 5')
print()
print('# Full training (local GPU, 256 envs, 500K steps)')
print('PYTHONPATH=src .venv/bin/python examples/rl/train_ppo_ik_curriculum.py \\')
print('    --envs 256 --total-steps 500000 --object ycb_013_apple')
print()
print('Available CLI arguments:')
print('  --envs         Number of parallel Genesis envs (default: 64)')
print('  --total-steps  Total training steps (default: 500000)')
print('  --object       YCB object name (default: ycb_013_apple)')
print('  --settle-steps Object settle steps (default: 30)')
print('  --seed         Random seed (default: 0)')
print('  --output-dir   Output directory (default: results/ppo_ik_curriculum)')
print('  --lr           Learning rate (default: 3e-4)')
print('  --n-steps      PPO rollout steps (default: 1024)')
print('  --batch-size   PPO batch size (default: 64)')
print('  --n-epochs     PPO epochs per update (default: 10)')

## まとめ / Summary

PPO + IKカリキュラムによる把持学習の流れを振り返ります / Review of PPO + IK curriculum grasp learning:

| Step | Component | Description |
|------|-----------|-------------|
| Curriculum | `curriculum.py` | 3-stage IK-to-policy blend manager / 3段階IK→ポリシー移行管理 |
| IK Planner | `ik_planner.py` | Precompute phase targets with CMA-ES params / CMA-ESパラメータで位相目標を事前計算 |
| RL Env | `hsr_pick_rl_env.py` | Batched Gymnasium env + VecEnv wrapper / バッチGymnasium環境 + VecEnvラッパー |
| Training | `train_ppo_ik_curriculum.py` | SB3 PPO with curriculum callback / カリキュラムコールバック付きSB3 PPO |
| Evaluation | `eval_ppo_policy.py` | Load checkpoint, eval per object / チェックポイント読み込み・物体別評価 |

### 重要なポイント / Key takeaways

- **残差ポリシー / Residual policy**: IKの上にNN補正を学習 — IKが粗い動き、ポリシーが微調整 / Learn NN corrections on top of IK — IK handles coarse motion, policy fine-tunes
- **カリキュラム学習 / Curriculum learning**: Stage 0 (純IK) → Stage 2 (ポリシー主導) への段階的移行が安定した学習を実現 / Gradual transition from Stage 0 (pure IK) to Stage 2 (policy-dominant) enables stable learning
- **バッチ環境 / Batched env**: N個のGenesis並列環境を1つのGymnasium環境として扱い、GPU利用率を最大化 / Treat N Genesis parallel envs as one Gymnasium env to maximize GPU utilization
- **CMA-ES + PPOの相乗効果 / CMA-ES + PPO synergy**: CMA-ESで最適化したパラメータをIKベースとして使用し、PPOがその上を改善 / CMA-ES optimized params provide IK base, PPO improves on top

### CMA-ES vs PPOの比較 / CMA-ES vs PPO comparison

| Aspect | CMA-ES (Section 9) | PPO + IK Curriculum (Section 10) |
|--------|---------------------|--------------------------------|
| What is learned | 4 grasp params per object | 9D residual policy (NN) |
| Search space | 28D (4 × 7 objects) | 32D obs → 9D action mapping |
| Generalization | Per-object only | Generalizes across placements |
| Training time | ~15 min (5 gen, popsize=32) | ~5 min (20K steps, 16 envs) |
| Adaptivity | Fixed params at runtime | Adapts per step based on state |

### 次のステップ / Next steps

- **マルチ物体学習 / Multi-object training**: 複数物体で同時に学習し、汎化ポリシーを獲得 / Train on multiple objects simultaneously for a generalized policy
- **視覚観測 / Visual observations**: カメラ画像を観測に追加し、視覚ベースの把持へ / Add camera images to observations for vision-based grasping
- **ドメインランダム化 / Domain randomization**: 摩擦・質量・照明をランダム化し、Sim-to-Real移行を準備 / Randomize friction, mass, lighting for Sim-to-Real transfer
- **戦略選択 / Strategy selection**: トップダウン・サイド・ピンチの把持戦略をポリシーが選択 / Policy selects grasp strategies (top, side, pinch)